<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/mmgbsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install rdkit pandas
!wget https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
!chmod +x gnina
!mv gnina /usr/local/bin/
!rm -f /usr/local/conda-meta/pinned
!mamba install -c conda-forge -c bioconda ambertools=23 python=3.11 -y
!mamba install -c conda-forge openbabel -y

--2026-07-21 16:33:57--  https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/45548146/6841601b-545d-4c9f-b4b1-f0eaa8ec7b74?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-21T17%3A20%3A17Z&rscd=attachment%3B+filename%3Dgnina&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-21T16%3A20%3A01Z&ske=2026-07-21T17%3A20%3A17Z&sks=b&skv=2018-11-09&sig=Ilc1xVtAnSom2g6fSCjkB6ApgdN%2BVgMPd5mUJfsl9xs%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NDY1NTIzOCwibmJmIjoxNzg0NjUxNjM4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2luZG93cy5uZ

In [3]:
import os
import re
from rdkit import Chem
from rdkit.Chem import rdMolTransforms


def generate_gnina_commands(sdf_filepath, output_script="run_gnina.sh"):
    """Reads an SDF of single 3D molecules, pre-centers them into the target box,

    calculates physiological net charge, routes to the correct receptor (mod2 vs
    mod3),
    and generates GNINA docking commands with automatic local backup.
    """
    mod2_receptor = "mod2.pdb"
    mod3_receptor = "mod3.pdb"

    # PyMOL calculated coordinates
    mod2_center = {"x": -4.2845716, "y": 0.39099988, "z": -2.0300002}
    mod3_center = {"x": 5.274572, "y": -0.47214293, "z": -1.3477145}

    # Box size
    box_size = 25.0

    # Ensure output directory exists
    os.makedirs("gnina_poses", exist_ok=True)

    supplier = Chem.SDMolSupplier(sdf_filepath)

    with open(output_script, "w") as f:
        f.write("#!/bin/bash\n")
        f.write("mkdir -p gnina_poses\n\n")

        valid_mols = 0
        for mol in supplier:
            if mol is None:
                continue

            # Extract and sanitize molecule ID
            mol_id_raw = (
                mol.GetProp("_Name")
                if mol.HasProp("_Name")
                else f"Mol_{valid_mols}"
            )
            mol_id = re.sub(r"[^a-zA-Z0-9_.-]", "_", mol_id_raw)
            mol_id = re.sub(r"_+", "_", mol_id).strip("_-")

            valid_mols += 1

            # Calculate formal charge to determine receptor and grid target
            net_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())

            if net_charge >= 1:
                receptor = mod2_receptor
                center = mod2_center
            else:
                receptor = mod3_receptor
                center = mod3_center

            try:
                conf = mol.GetConformer()
                centroid = rdMolTransforms.ComputeCentroid(conf)
                dx = center["x"] - centroid.x
                dy = center["y"] - centroid.y
                dz = center["z"] - centroid.z

                # Shift atom coordinates to box center
                for i in range(mol.GetNumAtoms()):
                    pos = conf.GetAtomPosition(i)
                    conf.SetAtomPosition(
                        i, (pos.x + dx, pos.y + dy, pos.z + dz)
                    )
            except Exception as e:
                print(f"Warning: Could not pre-center {mol_id}: {e}")

            # Save single isolated ligand input pose
            temp_ligand_path = f"gnina_poses/{mol_id}_input.sdf"
            out_pose_path = f"gnina_poses/{mol_id}_docked.sdf"

            writer = Chem.SDWriter(temp_ligand_path)
            writer.write(mol)
            writer.close()

            # Construct the GNINA command
            gnina_cmd = (
                f"gnina -r {receptor} -l {temp_ligand_path} "
                f"--center_x {center['x']:.3f} --center_y {center['y']:.3f} --center_z {center['z']:.3f} "
                f"--size_x {box_size} --size_y {box_size} --size_z {box_size} "
                f"--exhaustiveness 16 --num_modes 1 -o {out_pose_path} "
                f"--seed 42\n"
            )
            f.write(gnina_cmd)

        f.write("echo 'Docking finished. Creating local tar.gz backup...'\n")
        f.write("tar -czvf gnina_poses_backup.tar.gz gnina_poses/*_docked.sdf\n")
        f.write("Created: echo 'gnina_poses_backup.tar.gz'\n")

    print(f"GNINA batch script generated: {output_script} ({valid_mols} molecules)")
    os.chmod(output_script, 0o755)


print("Generating GNINA docking script for Training Actives...")
generate_gnina_commands(
    "Training_Actives_3D_1.sdf", output_script="run_gnina_actives.sh"
)

print("Generating GNINA docking script for Training Negatives...")
generate_gnina_commands(
    "Training_Negatives_3D_1.sdf", output_script="run_gnina_negatives.sh"
)

Generating GNINA docking script for Training Actives...
GNINA batch script generated: run_gnina_actives.sh (83 molecules)
Generating GNINA docking script for Training Negatives...
GNINA batch script generated: run_gnina_negatives.sh (54 molecules)


In [5]:
!bash run_gnina_actives.sh
!bash run_gnina_negatives.sh

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:e9cb230+   Built Feb 11 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: gnina -r mod3.pdb -l gnina_poses/methyl-Tryptophan_conf_0_input.sdf --center_x 5.275 --center_y -0.472 --center_z -1.348 --size_x 25.0 --size_y 25.0 --size_z 25.0 --exhaustiveness 16 --num_modes 1 -o gnina_poses/methyl-Tryptophan_conf_0_docked.sdf --seed 42
Using random seed: 42

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |    CNN     |   CNN
     | (kcal/mol) | pose score | affinity
-----+------------+------------+----------
    1       -5.51       0.7944      4.633
              _             
   

In [6]:
!tar -czvf docked_poses_backup.tar.gz gnina_poses/

gnina_poses/
gnina_poses/Z5105444405_conf_0_input.sdf
gnina_poses/Z10000553232_conf_0_input.sdf
gnina_poses/Z10001586075_conf_0_docked.sdf
gnina_poses/Z10000148697_conf_0_input.sdf
gnina_poses/Structure1_conf_0_input.sdf
gnina_poses/Z10001438277_conf_0_docked.sdf
gnina_poses/Z9667168424_conf_0_input.sdf
gnina_poses/cmpd10_active_conf_0_docked.sdf
gnina_poses/Structure1_conf_0_docked.sdf
gnina_poses/Structure6_conf_0_input.sdf
gnina_poses/Z10001431067_conf_0_docked.sdf
gnina_poses/Structure11_conf_0_input.sdf
gnina_poses/Z10044722512_conf_0_docked.sdf
gnina_poses/Mol_1_conf_0_input.sdf
gnina_poses/Structure30_conf_0_docked.sdf
gnina_poses/Z3832048189_conf_0_docked.sdf
gnina_poses/Z10000572257_conf_0_docked.sdf
gnina_poses/Structure26_conf_0_docked.sdf
gnina_poses/Z10001467695_conf_0_docked.sdf
gnina_poses/Z10001467695_conf_0_input.sdf
gnina_poses/Z3347345587_conf_0_docked.sdf
gnina_poses/Structure13_conf_0_docked.sdf
gnina_poses/Mol_13_conf_0_input.sdf
gnina_poses/Mol_11_conf_0_input.sd

In [27]:
def remove_hydrogens(input_pdb, output_pdb):
    with open(input_pdb, 'r') as infile, open(output_pdb, 'w') as outfile:
        for line in infile:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                atom_name = line[12:16].strip()
                # Skip the line if the atom name starts with 'H' or a number followed by 'H' (e.g., '1H')
                if atom_name.startswith('H') or (atom_name[0].isdigit() and atom_name[1] == 'H'):
                    continue
            outfile.write(line)

print("Cleaning receptors...")
remove_hydrogens("mod2.pdb", "mod2_clean.pdb")
remove_hydrogens("mod3.pdb", "mod3_clean.pdb")
print("Done! Cleaned PDBs generated.")

Cleaning receptors...
Done! Cleaned PDBs generated.


In [31]:
import os
import glob
import shutil
import subprocess
from rdkit import Chem

def setup_mmpbsa():
    print("Setting up MM/GBSA pipeline...")

    antechamber_path = shutil.which("antechamber")
    parmchk2_path = shutil.which("parmchk2")
    tleap_path = shutil.which("tleap")
    mmpbsa_path = shutil.which("MMPBSA.py")
    obabel_path = shutil.which("obabel")

    if not all([antechamber_path, parmchk2_path, tleap_path, mmpbsa_path, obabel_path]):
        print("ERROR: Missing AmberTools or OpenBabel in standard paths.")
        return

    os.makedirs("mmpbsa_run", exist_ok=True)

    mmpbsa_in = """&general
   startframe=1, endframe=1, interval=1,
   keep_files=0,
/
&gb
   igb=2, saltcon=0.150,
/
"""
    with open("mmpbsa_run/mmpbsa.in", "w") as f:
        f.write(mmpbsa_in)

    output_script = "run_mmpbsa.sh"
    sdf_files = glob.glob("gnina_poses/*_docked.sdf") + glob.glob("../gnina_poses/*_docked.sdf")

    if not sdf_files:
        print("Error: No docked SDF files found.")
        return

    with open(output_script, 'w') as f:
        f.write("#!/bin/bash\n")
        f.write("export AMBERHOME=\"/usr/local\"\n")
        f.write("cd mmpbsa_run\n\n")

        for sdf in sdf_files:
            abs_sdf = os.path.abspath(sdf)
            base_name = os.path.basename(sdf)
            mol_id = base_name.replace("_docked.sdf", "")

            try:
                # Extract net charge with RDKit
                supplier = Chem.SDMolSupplier(abs_sdf, sanitize=False, removeHs=False)
                if not supplier or len(supplier) == 0:
                    continue
                mol = supplier[0]
                if mol is None:
                    continue

                mol.UpdatePropertyCache(strict=False)
                net_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())

                # Add 3D hydrogens safely with OpenBabel
                fixed_sdf_name = f"{mol_id}_fixed.sdf"
                fixed_sdf_path = os.path.join("mmpbsa_run", fixed_sdf_name)

                obabel_cmd = [obabel_path, "-isdf", abs_sdf, "-osdf", "-O", fixed_sdf_path, "-h"]
                subprocess.run(obabel_cmd, check=True, capture_output=True)

            except Exception as e:
                print(f"Warning: Processing failed for {mol_id}. Skipping.")
                continue

            # Route receptor based on charge
            receptor_file = "mod2_clean.pdb" if net_charge >= 1 else "mod3_clean.pdb"
            abs_receptor = os.path.abspath(receptor_file)

            f.write(f"echo 'Running MM/GBSA for {mol_id}'\n")

            f.write(f"if {antechamber_path} -i {mol_id}_fixed.sdf -fi sdf -o {mol_id}_lig.mol2 -fo mol2 -c bcc -s 2 -nc {net_charge}; then\n")
            f.write(f"  {parmchk2_path} -i {mol_id}_lig.mol2 -f mol2 -o {mol_id}_lig.frcmod\n")

            tleap_script = f"{mol_id}_tleap.in"
            f.write(f"  cat <<EOF > {tleap_script}\n")
            f.write("source leaprc.protein.ff14SB\n")
            f.write("source leaprc.water.tip3p\n")  # Adds parameter definitions for NA / CL ions
            f.write("source leaprc.gaff2\n")
            f.write("set default PBRadii mbondi2\n")
            f.write(f"REC = loadpdb {abs_receptor}\n")
            f.write(f"LIG = loadmol2 {mol_id}_lig.mol2\n")
            f.write(f"loadamberparams {mol_id}_lig.frcmod\n")
            f.write("COMP = combine { REC LIG }\n")  # Added essential spaces inside braces
            f.write(f"saveamberparm COMP {mol_id}_complex.prmtop {mol_id}_complex.inpcrd\n")
            f.write(f"saveamberparm REC {mol_id}_receptor.prmtop {mol_id}_receptor.inpcrd\n")
            f.write(f"saveamberparm LIG {mol_id}_ligand.prmtop {mol_id}_ligand.inpcrd\n")
            f.write("quit\nEOF\n")

            f.write(f"  {tleap_path} -f {tleap_script} > {mol_id}_tleap.log 2>&1\n")

            f.write(f"  if [ -f {mol_id}_complex.prmtop ]; then\n")
            f.write(f"    {mmpbsa_path} -O -i mmpbsa.in -o {mol_id}_FINAL_DELTA_G.dat "
                    f"-cp {mol_id}_complex.prmtop "
                    f"-rp {mol_id}_receptor.prmtop -lp {mol_id}_ligand.prmtop "
                    f"-y {mol_id}_complex.inpcrd\n")
            f.write("  else\n")
            f.write(f"    echo 'Tleap failed for {mol_id}. Check {mol_id}_tleap.log'\n")
            f.write("  fi\n")

            f.write("else\n")
            f.write(f"  echo 'Antechamber rejected {mol_id}. Skipping...'\n")
            f.write("fi\n\n")

    print(f"MM/GBSA batch script successfully updated. Run via terminal: !bash {output_script}")

setup_mmpbsa()

Setting up MM/GBSA pipeline...


[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:29] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

MM/GBSA batch script successfully updated. Run via terminal: !bash run_mmpbsa.sh


[17:15:32] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:32] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:15:32] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [35]:
!bash run_mmpbsa.sh

Streaming output truncated to the last 5000 lines.
Info: Total number of electrons: 186; net charge: 0

Running: /usr/local/bin/sqm -O -i sqm.in -o sqm.out

Running: /usr/local/bin/am1bcc -i ANTECHAMBER_AM1BCC_PRE.AC -o ANTECHAMBER_AM1BCC.AC -f ac -p /usr/local/dat/antechamber/BCCPARM.DAT -s 2 -j 1

Running: /usr/local/bin/atomtype -f ac -p bcc -o ANTECHAMBER_AM1BCC.AC -i ANTECHAMBER_AM1BCC_PRE.AC

Loading and checking parameter files for compatibility...
cpptraj found! Using /usr/local/bin/cpptraj
mmpbsa_py_energy found! Using /usr/local/bin/mmpbsa_py_energy
Preparing trajectories for simulation...
1 frames were processed by cpptraj for use in calculation.

Running calculations on normal system...

Beginning GB calculations with /usr/local/bin/mmpbsa_py_energy
  calculating complex contribution...
  calculating receptor contribution...
  calculating ligand contribution...

Timing:
Total setup time:                           1.148 sec.
Creating trajectories with cpptraj:         0.317 

In [39]:
import os
import glob
import pandas as pd

def compile_mmgbsa_results():
    print("Scanning for MM/GBSA result files...")

    dat_files = glob.glob("mmpbsa_run/*_FINAL_DELTA_G.dat")

    if not dat_files:
        print("Error: No _FINAL_DELTA_G.dat files found in the 'mmpbsa_run' folder.")
        return

    results = []

    for file_path in dat_files:
        filename = os.path.basename(file_path)
        mol_name = filename.replace("_FINAL_DELTA_G.dat", "")
        delta_g = None

        with open(file_path, 'r') as f:
            lines = f.readlines()
            in_differences_section = False

            for line in lines:
                # Jump straight to the final math section
                if "Differences (Complex - Receptor - Ligand):" in line:
                    in_differences_section = True

                # Grab the DELTA TOTAL row
                if in_differences_section and line.startswith("DELTA TOTAL"):
                    parts = line.split()
                    # The line looks like: DELTA TOTAL   -45.32   2.11   0.55
                    # So the value we want is the 3rd item in the list (index 2)
                    if len(parts) >= 3:
                        try:
                            delta_g = float(parts[2])
                        except ValueError:
                            pass
                    break

        if delta_g is not None:
            results.append({"Molecule": mol_name, "Delta_G (kcal/mol)": delta_g})
        else:
            print(f"Warning: Could not parse DELTA TOTAL in {filename}")

    if not results:
        print("No valid Delta G values were successfully extracted.")
        return

    # Convert to DataFrame, sort, and export
    df = pd.DataFrame(results)
    df = df.sort_values(by="Delta_G (kcal/mol)", ascending=True).reset_index(drop=True)
    output_file = "Ranked_MMGBSA_Results.csv"
    df.to_csv(output_file, index=False)

    print(f"\nExtraction complete. Results saved to '{output_file}'\n")
    print(df.to_string(index=False))

# Run the function
compile_mmgbsa_results()

Scanning for MM/GBSA result files...

Extraction complete. Results saved to 'Ranked_MMGBSA_Results.csv'

                Molecule  Delta_G (kcal/mol)
     cmpd1_active_conf_0       -2.133490e+01
     cmpd7_active_conf_0       -1.574480e+01
      Structure26_conf_0       -1.526960e+01
      Structure18_conf_0       -1.514300e+01
     Z10044158896_conf_0       -1.388870e+01
       Structure6_conf_0       -1.358720e+01
      Structure29_conf_0       -1.268030e+01
      Structure22_conf_0       -1.267230e+01
     Z10001586095_conf_0       -1.179770e+01
       Structure4_conf_0       -1.118170e+01
      Structure21_conf_0       -1.112030e+01
      Structure23_conf_0       -1.110050e+01
     cmpd3_active_conf_0       -1.051940e+01
     Z10000541815_conf_0       -1.040810e+01
      Structure16_conf_0       -9.290700e+00
      Structure24_conf_0       -8.961700e+00
     Z10000541814_conf_0       -5.717400e+00
       Structure1_conf_0       -5.334800e+00
    cmpd12_active_conf_0       -4.615100

In [40]:
cat mmpbsa_run/Z10001438277_conf_0_tleap.log

-I: Adding /usr/local/dat/leap/prep to search path.
-I: Adding /usr/local/dat/leap/lib to search path.
-I: Adding /usr/local/dat/leap/parm to search path.
-I: Adding /usr/local/dat/leap/cmd to search path.
-f: Source Z10001438277_conf_0_tleap.in.

Welcome to LEaP!
(no leaprc in search path)
Sourcing: ./Z10001438277_conf_0_tleap.in
----- Source: /usr/local/dat/leap/cmd/leaprc.protein.ff14SB
----- Source of /usr/local/dat/leap/cmd/leaprc.protein.ff14SB done
Log file: ./leap.log
Loading parameters: /usr/local/dat/leap/parm/parm10.dat
Reading title:
PARM99 + frcmod.ff99SB + frcmod.parmbsc0 + OL3 for RNA
Loading parameters: /usr/local/dat/leap/parm/frcmod.ff14SB
Reading force field modification type file (frcmod)
Reading title:
ff14SB protein backbone and sidechain parameters
Loading library: /usr/local/dat/leap/lib/amino12.lib
Loading library: /usr/local/dat/leap/lib/aminoct12.lib
Loading library: /usr/local/dat/leap/lib/aminont12.lib
----- Source: /usr/local/dat/leap/cmd/leaprc.water.tip3